# Notebook 05 - Modeling

Notebook này thực hiện quá trình xây dựng, huấn luyện và đánh giá các mô hình học máy nhằm dự đoán mức độ Digital Burnout của người học dựa trên các chỉ báo đã được xác thực ở Notebook 04.

Khác với Notebook 04 chỉ tập trung đánh giá giá trị thống kê của các chỉ báo, notebook này sử dụng các chỉ báo đó để xây dựng các mô hình dự đoán, so sánh hiệu năng giữa các thuật toán và lựa chọn mô hình tối ưu phục vụ cho bước giải thích mô hình (Interpretability) ở notebook tiếp theo.

Notebook bao gồm các bước chính sau:

1. Chuẩn bị dữ liệu cho mô hình học máy.
2. Xây dựng quy trình tiền xử lý dữ liệu.
3. Huấn luyện các mô hình học máy.
4. Đánh giá hiệu năng mô hình bằng nhiều chỉ số.
5. So sánh các mô hình và lựa chọn mô hình tốt nhất.
6. Tổng hợp các nhận xét phục vụ thảo luận nghiên cứu.

---

### Input

- `digital_burnout_cleaned.csv`
- `validated_indicators.csv` 

---

### Output

- Kết quả Cross Validation
- Kết quả đánh giá trên Validation Set
- Kết quả đánh giá trên Test Set
- Bảng so sánh các mô hình
- Mô hình được lựa chọn cho Notebook 06 – Interpretability

# 0. Set Up

Phần này chuẩn bị môi trường làm việc cho toàn bộ notebook. Các thư viện cần thiết sẽ được import, các tham số chung được thiết lập và hệ thống sẽ kiểm tra khả năng sử dụng GPU nhằm tối ưu thời gian huấn luyện mô hình.

Việc thiết lập thống nhất ngay từ đầu giúp đảm bảo khả năng tái lập kết quả (Reproducibility) và tính nhất quán trong toàn bộ quá trình nghiên cứu.

In [1]:
# Import các thư viện xử lý dữ liệu

import gc
import os
import time
import warnings

import joblib
import numpy as np
import pandas as pd

# Import thư viện trực quan hóa

import matplotlib.pyplot as plt
import seaborn as sns

# Import các thành phần của Scikit-learn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    OrdinalEncoder
)

# Tắt các cảnh báo không cần thiết

warnings.filterwarnings("ignore")

In [2]:
# Kiểm tra thư viện XGBoost

try:

    from xgboost import XGBClassifier

    xgboost_available = True

except ImportError:

    xgboost_available = False

In [3]:
# Thiết lập các tham số dùng chung

RANDOM_STATE = 42

TEST_SIZE = 0.15

VALIDATION_SIZE = 0.1765

print("Thiết lập môi trường hoàn tất.")
print()
print(f"Random state: {RANDOM_STATE}")
print(f"Tỷ lệ Validation: 15%")
print(f"Tỷ lệ Test: 15%")

Thiết lập môi trường hoàn tất.

Random state: 42
Tỷ lệ Validation: 15%
Tỷ lệ Test: 15%


In [5]:
# Kiểm tra khả năng sử dụng GPU cho XGBoost

gpu_available = False

if xgboost_available:

    try:

        test_model = XGBClassifier(
            tree_method="hist",
            device="cuda",
            n_estimators=1,
            verbosity=0
        )

        gpu_available = True

    except Exception:

        gpu_available = False

print(f"XGBoost khả dụng: {xgboost_available}")
print(f"GPU khả dụng: {gpu_available}")

XGBoost khả dụng: True
GPU khả dụng: True


# 1. Data Preparation

Phần này chuẩn bị toàn bộ dữ liệu đầu vào cho quá trình xây dựng mô hình học máy.

Các công việc bao gồm:

- Đọc dữ liệu đã làm sạch
- Đọc danh sách chỉ báo đã được xác thực
- Xây dựng tập đặc trưng và biến mục tiêu
- Chia dữ liệu thành tập huấn luyện, tập xác thực và tập kiểm tra
- Tiền xử lý dữ liệu
- Chuyển đổi dữ liệu sang định dạng phù hợp để huấn luyện mô hình

Sau khi hoàn thành phần này, toàn bộ mô hình sẽ sử dụng cùng một tập dữ liệu đã được tiền xử lý nhằm đảm bảo tính nhất quán trong quá trình thực nghiệm.

## 1.1 Load Dataset

Tải bộ dữ liệu đã được làm sạch từ Notebook 02 để sử dụng trong quá trình xây dựng mô hình học máy.

In [6]:
# Khai báo đường dẫn đến bộ dữ liệu đã làm sạch

cleaned_data_path = "../../data/processed/international_dataset/validated_dataset.csv"

In [8]:
# Đọc bộ dữ liệu đã làm sạch

df = pd.read_csv(cleaned_data_path)

print("Đã tải bộ dữ liệu đã làm sạch.")
print(f"Số lượng quan sát: {df.shape[0]:,}")
print(f"Số lượng biến: {df.shape[1]}")

Đã tải bộ dữ liệu đã làm sạch.
Số lượng quan sát: 5,000,000
Số lượng biến: 10


In [9]:
# Hiển thị một số quan sát đầu tiên

display(df.head())

,emotional_exhaustion,stress_level,daily_screen_time,doomscrolling_duration,notification_count,sleep_hours,burnout_score,burnout_risk,productivity_score,productivity_category
0,4,10,8.8,1.2,112,5.9,46,1,100,High
1,9,5,10.3,2.4,168,5.6,57,1,96,High
2,2,4,6.5,1.0,199,5.5,29,0,79,High
3,5,1,9.6,0.1,122,6.1,57,1,63,Medium
4,9,4,13.3,1.9,73,7.9,64,2,89,High


## 1.2 Feature matrix and target definition

Phần này xây dựng tập đặc trưng đầu vào và biến mục tiêu phục vụ cho quá trình huấn luyện mô hình.

Tập đặc trưng được tạo từ các chỉ báo đã được xác thực ở Notebook 04, trong khi biến mục tiêu là mức độ Digital Burnout (`burnout_risk`).

Việc sử dụng các chỉ báo đã được kiểm chứng giúp đảm bảo tính nhất quán giữa quá trình xác thực đặc trưng và quá trình xây dựng mô hình.

In [10]:
feature_columns = [

    column

    for column in df.columns

    if column not in [

        "burnout_score",
        "burnout_risk",
        "productivity_score",
        "productivity_category"

    ]

]

X = df[feature_columns]

y = df["burnout_risk"]

## 1.4 Train / Validation / Test Split

Sau khi xây dựng ma trận đặc trưng và biến mục tiêu, dữ liệu sẽ được chia thành ba tập độc lập bao gồm tập huấn luyện (Training Set), tập xác thực (Validation Set) và tập kiểm tra (Test Set).

Việc chia dữ liệu theo tỷ lệ **70% - 15% - 15%** giúp đảm bảo:

- Tập huấn luyện được sử dụng để xây dựng mô hình.
- Tập xác thực được sử dụng để so sánh và lựa chọn mô hình tốt nhất.
- Tập kiểm tra chỉ được sử dụng một lần cuối cùng nhằm đánh giá khả năng tổng quát hóa của mô hình.

Để duy trì tỷ lệ phân bố của các lớp trong biến mục tiêu, phương pháp **Stratified Sampling** được áp dụng trong cả hai lần chia dữ liệu.

In [11]:
# Chia dữ liệu thành tập huấn luyện và tập tạm thời

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

In [12]:
# Tiếp tục chia tập tạm thời thành Validation và Test

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

In [13]:
# Tổng hợp kích thước các tập dữ liệu

split_summary = pd.DataFrame(
    {
        "Dataset": [
            "Training Set",
            "Validation Set",
            "Test Set"
        ],
        "Samples": [
            len(X_train),
            len(X_val),
            len(X_test)
        ],
        "Percentage": [
            len(X_train) / len(X) * 100,
            len(X_val) / len(X) * 100,
            len(X_test) / len(X) * 100
        ]
    }
)

split_summary["Percentage"] = (
    split_summary["Percentage"]
    .round(2)
)

print("Kết quả chia bộ dữ liệu:")

display(split_summary)

Kết quả chia bộ dữ liệu:


,Dataset,Samples,Percentage
0,Training Set,3500000,70.0
1,Validation Set,750000,15.0
2,Test Set,750000,15.0


In [14]:
# Kiểm tra phân bố của biến mục tiêu trên từng tập dữ liệu

distribution_summary = pd.DataFrame(
    {
        "Training Set": (
            y_train.value_counts(normalize=True)
            .sort_index()
        ),
        "Validation Set": (
            y_val.value_counts(normalize=True)
            .sort_index()
        ),
        "Test Set": (
            y_test.value_counts(normalize=True)
            .sort_index()
        )
    }
)

distribution_summary = (
    distribution_summary
    .mul(100)
    .round(2)
)

print("Phân bố của biến mục tiêu trên từng tập dữ liệu (%):")

display(distribution_summary)

Phân bố của biến mục tiêu trên từng tập dữ liệu (%):


,Training Set,Validation Set,Test Set
burnout_risk,,,
0,34.75,34.75,34.75
1,32.19,32.19,32.19
2,33.06,33.06,33.06


In [15]:
# Hiển thị kích thước của từng tập dữ liệu

print("Thông tin các tập dữ liệu")
print()

print(f"Training Set: {X_train.shape}")
print(f"Validation Set: {X_val.shape}")
print(f"Test Set: {X_test.shape}")

Thông tin các tập dữ liệu

Training Set: (3500000, 6)
Validation Set: (750000, 6)
Test Set: (750000, 6)


In [16]:
print(f"Training Set Memory: {X_train.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Validation Set Memory: {X_val.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Test Set Memory: {X_test.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Training Set Memory: 186.92 MB
Validation Set Memory: 40.05 MB
Test Set Memory: 40.05 MB


## 1.5 Build preprocessing pipeline

Phần này xây dựng quy trình tiền xử lý dữ liệu trước khi đưa vào các mô hình học máy.

Quy trình tiền xử lý được thiết kế riêng cho từng kiểu dữ liệu nhằm đảm bảo tính nhất quán giữa các mô hình và hạn chế hiện tượng Data Leakage.

Đối với biến số, dữ liệu được xử lý bằng phương pháp thay thế giá trị khuyết theo trung vị và chuẩn hóa.

Đối với biến phân loại, nghiên cứu sử dụng phương pháp thay thế giá trị xuất hiện nhiều nhất kết hợp với Ordinal Encoding. So với One-Hot Encoding, phương pháp này giúp giảm đáng kể lượng bộ nhớ sử dụng khi xử lý bộ dữ liệu lớn, đồng thời vẫn đảm bảo khả năng học của các mô hình được lựa chọn trong nghiên cứu.

Sau khi xây dựng quy trình tiền xử lý, bộ tiền xử lý sẽ được huấn luyện trên tập Training và áp dụng cho tập Validation và Test nhằm đảm bảo tính nhất quán trong toàn bộ quá trình thực nghiệm.

In [17]:
# Xây dựng quy trình tiền xử lý cho các biến số

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

print("Đã xây dựng quy trình tiền xử lý cho biến số.")

Đã xây dựng quy trình tiền xử lý cho biến số.


In [18]:
# Xây dựng quy trình tiền xử lý cho các biến phân loại

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
        )
    ]
)

print("Đã xây dựng quy trình tiền xử lý cho biến phân loại.")

Đã xây dựng quy trình tiền xử lý cho biến phân loại.


In [19]:
# Xác định nhóm đặc trưng

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print(f"Numerical Features: {len(numerical_features)}")

print(f"Categorical Features: {len(categorical_features)}")

Numerical Features: 6
Categorical Features: 0


In [20]:
# Kết hợp các quy trình tiền xử lý

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

print("Đã xây dựng bộ tiền xử lý.")

Đã xây dựng bộ tiền xử lý.


In [21]:
# Huấn luyện bộ tiền xử lý trên tập Training

preprocessor.fit(X_train)

print("Đã huấn luyện bộ tiền xử lý.")

Đã huấn luyện bộ tiền xử lý.


In [22]:
# Chuyển đổi tập Training

X_train_processed = preprocessor.transform(X_train)

# Chuyển đổi tập Validation

X_val_processed = preprocessor.transform(X_val)

# Chuyển đổi tập Test

X_test_processed = preprocessor.transform(X_test)

print("Đã hoàn thành quá trình tiền xử lý dữ liệu.")

Đã hoàn thành quá trình tiền xử lý dữ liệu.


In [23]:
# Hiển thị kích thước dữ liệu sau khi tiền xử lý

print("Kích thước dữ liệu sau tiền xử lý")
print()

print(f"Training Set: {X_train_processed.shape}")
print(f"Validation Set: {X_val_processed.shape}")
print(f"Test Set: {X_test_processed.shape}")

Kích thước dữ liệu sau tiền xử lý

Training Set: (3500000, 6)
Validation Set: (750000, 6)
Test Set: (750000, 6)


In [24]:
# Kiểm tra kiểu dữ liệu sau khi tiền xử lý

print("Kiểu dữ liệu sau tiền xử lý:")

print(type(X_train_processed))

Kiểu dữ liệu sau tiền xử lý:
<class 'numpy.ndarray'>


In [25]:
# Giải phóng bộ nhớ của các biến không còn sử dụng

gc.collect()

print("Đã giải phóng bộ nhớ tạm.")

Đã giải phóng bộ nhớ tạm.


# 2. Model Development

Phần này xây dựng các mô hình học máy nhằm dự đoán mức độ Digital Burnout dựa trên các chỉ báo đã được xác thực.

Các mô hình được lựa chọn bao gồm:

- Mô hình cơ sở (Baseline Model)
- Các mô hình học máy truyền thống (Traditional Machine Learning Models)
- Các mô hình kết hợp (Hybrid Models)

Tất cả các mô hình được huấn luyện trên cùng một tập dữ liệu đã được tiền xử lý nhằm đảm bảo tính nhất quán trong quá trình so sánh hiệu năng.

## 2.1 Model selection

Phần này khởi tạo các mô hình sẽ được sử dụng trong nghiên cứu.

Các mô hình được lựa chọn đại diện cho ba nhóm thuật toán khác nhau nhằm đánh giá khả năng dự đoán Digital Burnout dưới nhiều góc độ.

Các mô hình bao gồm:

- Logistic Regression (Baseline)
- Random Forest (Traditional)
- XGBoost (Traditional)
- Soft Voting Ensemble (Hybrid)
- Stacking Ensemble (Hybrid)

Việc khởi tạo mô hình được thực hiện trước quá trình huấn luyện nhằm giúp notebook dễ bảo trì và thuận tiện cho việc điều chỉnh siêu tham số.

In [26]:
# Khởi tạo mô hình Logistic Regression

logistic_model = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    n_jobs=-1
)

In [27]:
# Khởi tạo mô hình Random Forest

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=2
)

In [28]:
# Khởi tạo mô hình XGBoost

if xgboost_available:
    
    xgboost_model = XGBClassifier(
        
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        device="cuda",
        n_estimators=120,
        learning_rate=0.1,
        max_depth=5,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        enable_categorical=False
    )

In [29]:
# Khởi tạo mô hình Soft Voting

voting_model = VotingClassifier(
    estimators=[
        ("lr", logistic_model),
        ("rf", random_forest_model),
        ("xgb", xgboost_model)
    ],
    voting="soft",
    n_jobs=2
)

In [30]:
# Khởi tạo mô hình Stacking

stacking_model = StackingClassifier(
    estimators=[
        ("rf", random_forest_model),
        ("xgb", xgboost_model)
    ],
    final_estimator=LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000
    ),
    cv=3,
    n_jobs=2
)

In [31]:
# Tổng hợp các mô hình sử dụng trong nghiên cứu

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "XGBoost": xgboost_model,
    "Soft Voting": voting_model,
    "Stacking": stacking_model
}

print("Các mô hình được khởi tạo thành công.")
print()

for model_name in models.keys():
    print(f"- {model_name}")

Các mô hình được khởi tạo thành công.

- Logistic Regression
- Random Forest
- XGBoost
- Soft Voting
- Stacking


## 2.2 Model training function

Phần này xây dựng một hàm dùng chung để huấn luyện và đánh giá các mô hình học máy.

Thay vì lặp lại cùng một đoạn mã cho từng mô hình, nghiên cứu sử dụng một hàm thống nhất nhằm đảm bảo quy trình huấn luyện được thực hiện nhất quán giữa tất cả các thuật toán.

Đối với mỗi mô hình, hàm sẽ thực hiện các công việc sau:

- Huấn luyện mô hình trên tập Training
- Ghi nhận thời gian huấn luyện
- Dự đoán trên tập Validation
- Tính xác suất dự đoán
- Lưu toàn bộ kết quả phục vụ các bước đánh giá tiếp theo

Việc chỉ huấn luyện một mô hình tại mỗi thời điểm cũng giúp giảm đáng kể lượng bộ nhớ sử dụng đối với bộ dữ liệu lớn.

In [32]:
# Khởi tạo nơi lưu kết quả của các mô hình

trained_models = {}

model_predictions = {}

model_probabilities = {}

model_results = {}

Do bộ dữ liệu nghiên cứu có quy mô rất lớn, nghiên cứu áp dụng chiến lược huấn luyện hai giai đoạn.

Trong giai đoạn lựa chọn mô hình, mỗi thuật toán được huấn luyện trên tập dữ liệu gồm tối đa 1.000.000 mẫu được lấy ngẫu nhiên theo phương pháp phân tầng (Stratified Sampling). Cách tiếp cận này giúp giảm đáng kể thời gian thực nghiệm và chi phí tính toán, đồng thời vẫn duy trì được phân bố của biến mục tiêu.

Sau khi xác định được mô hình có hiệu năng tốt nhất, mô hình đó sẽ được huấn luyện lại trên toàn bộ tập Training ở bước cuối của notebook trước khi lưu mô hình. Điều này đảm bảo mô hình cuối cùng tận dụng đầy đủ dữ liệu để đạt hiệu năng tối ưu.

In [33]:
# Xây dựng hàm huấn luyện mô hình

def train_single_model(model_name, model):

    print()
    print(f"Đang huấn luyện mô hình: {model_name}")

    # Chỉ sử dụng tối đa 1 triệu mẫu cho giai đoạn lựa chọn mô hình

    max_training_samples = 1_000_000

    if X_train_processed.shape[0] > max_training_samples:

        X_train_subset, _, y_train_subset, _ = train_test_split(
            X_train_processed,
            y_train,
            train_size=max_training_samples,
            stratify=y_train,
            random_state=RANDOM_STATE
        )

    else:

        X_train_subset = X_train_processed
        y_train_subset = y_train

    print(f"Số mẫu sử dụng để huấn luyện: {len(y_train_subset):,}")

    start_time = time.perf_counter()

    # Huấn luyện mô hình

    if model_name == "XGBoost":

        model.fit(
            X_train_subset,
            y_train_subset,
            eval_set=[
                (X_val_processed, y_val)
            ],
            verbose=False
        )

    else:

        model.fit(
            X_train_subset,
            y_train_subset
        )

    training_time = time.perf_counter() - start_time

    # Dự đoán trên tập Validation

    y_pred = model.predict(
        X_val_processed
    )

    # Dự đoán xác suất

    if hasattr(model, "predict_proba"):

        y_prob = model.predict_proba(
            X_val_processed
        )[:, 1]

    else:

        y_prob = None

    # Lưu mô hình

    trained_models[model_name] = model

    # Lưu kết quả dự đoán

    model_predictions[model_name] = y_pred

    model_probabilities[model_name] = y_prob

    # Lưu thông tin huấn luyện

    model_results[model_name] = {
        "Training Samples": X_train_subset.shape[0],
        "Features": X_train_subset.shape[1],
        "Training Time": training_time
    }

    print(f"Huấn luyện hoàn tất trong {training_time:.2f} giây.")

    del X_train_subset
    del y_train_subset

    gc.collect()

In [34]:
# Kiểm tra hàm huấn luyện đã được khởi tạo

print("Đã khởi tạo hàm huấn luyện mô hình.")

Đã khởi tạo hàm huấn luyện mô hình.


## 2.3 Train baseline model

Phần này huấn luyện mô hình cơ sở (Baseline Model) của nghiên cứu.

Logistic Regression được lựa chọn làm mô hình cơ sở nhằm cung cấp một mốc tham chiếu cho việc đánh giá hiệu quả của các mô hình học máy truyền thống và các mô hình kết hợp.

Kết quả của mô hình này sẽ được sử dụng để so sánh với các mô hình phức tạp hơn trong các phần tiếp theo.

In [35]:
# Huấn luyện mô hình Logistic Regression

train_single_model(
    model_name="Logistic Regression",
    model=logistic_model
)


Đang huấn luyện mô hình: Logistic Regression
Số mẫu sử dụng để huấn luyện: 1,000,000
Huấn luyện hoàn tất trong 5.82 giây.


In [36]:
# Hiển thị thông tin sau khi huấn luyện

print("Thông tin mô hình Logistic Regression")
print()

display(
    pd.DataFrame(
        [model_results["Logistic Regression"]]
    )
)

Thông tin mô hình Logistic Regression



,Training Samples,Features,Training Time
0,1000000,6,5.815042


In [37]:
# Kiểm tra kích thước dữ liệu dự đoán

print("Kích thước kết quả dự đoán")
print()

print(
    f"Số lượng dự đoán: {len(model_predictions['Logistic Regression']):,}"
)

print(
    f"Số lượng xác suất: {len(model_probabilities['Logistic Regression']):,}"
)

Kích thước kết quả dự đoán

Số lượng dự đoán: 750,000
Số lượng xác suất: 750,000


In [38]:
# Giải phóng bộ nhớ tạm

gc.collect()

print("Đã giải phóng bộ nhớ.")

Đã giải phóng bộ nhớ.


## 2.4 Train traditional models

Phần này huấn luyện các mô hình học máy truyền thống nhằm đánh giá khả năng dự đoán Digital Burnout trên cùng một tập dữ liệu.

Hai mô hình được lựa chọn bao gồm:

- Random Forest
- XGBoost

Các mô hình này đều được huấn luyện trên tập Training và đánh giá trên tập Validation để đảm bảo tính nhất quán trong quá trình so sánh.

In [39]:
# Huấn luyện mô hình Random Forest

train_single_model(
    model_name="Random Forest",
    model=random_forest_model
)


Đang huấn luyện mô hình: Random Forest
Số mẫu sử dụng để huấn luyện: 1,000,000
Huấn luyện hoàn tất trong 152.73 giây.


In [40]:
# Hiển thị thông tin của mô hình Random Forest

print("Thông tin mô hình Random Forest")
print()

display(
    pd.DataFrame(
        [model_results["Random Forest"]]
    )
)

Thông tin mô hình Random Forest



,Training Samples,Features,Training Time
0,1000000,6,152.728712


In [41]:
# Giải phóng bộ nhớ sau khi huấn luyện

gc.collect()

print("Đã giải phóng bộ nhớ.")

Đã giải phóng bộ nhớ.


In [42]:
# Huấn luyện mô hình XGBoost

train_single_model(
    model_name="XGBoost",
    model=xgboost_model
)


Đang huấn luyện mô hình: XGBoost
Số mẫu sử dụng để huấn luyện: 1,000,000


XGBoostError: [20:45:35] C:\actions-runner\_work\xgboost\xgboost\src\metric\elementwise_metric.cu:331: Check failed: preds.Size() == info.labels.Size() (2250000 vs. 750000) : label and prediction size not match, hint: use merror or mlogloss for multi-class classification

In [ ]:
# Hiển thị thông tin của mô hình XGBoost

print("Thông tin mô hình XGBoost")
print()

display(
    pd.DataFrame(
        [model_results["XGBoost"]]
    )
)

In [ ]:
# Giải phóng bộ nhớ sau khi huấn luyện

gc.collect()

print("Đã giải phóng bộ nhớ.")

Đã giải phóng bộ nhớ.
